# Molecular formula and delta-formula annotation (SIRIUS + BUDDY)

For every core: keep BUDDY formulas with estimated FDR < 0.05 and SIRIUS formulas with SiriusScoreNormalized > 0.9 and ≥ 5 explained peaks; subtract the neutral core formula with `chemparse` to obtain the delta formula; build the library `annotation` string. C28 bile alcohols keep only the predicted formulas (no core formula to subtract).

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import pandas as pd
import chemparse
import re

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
BASE = f'{PROJECT_DIR}'
OUTPUT_DIR = f'{BASE}/Scripts_data_analysis/library_generation'

# One entry per core. delta_ref / ba_formula = None means "no delta" (C28):
# those cores only keep the SIRIUS and BUDDY predicted formulas.
CONFIG = {
    'c24_nonhydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_nonhydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/c24_nonhydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_nonhydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O2', 'delta_ref': 361.3101,
    },
    'c24_monohydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_monohydroxy_output_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/c24_monohydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_monohydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O3', 'delta_ref': 377.305,
    },
    'c24_dihydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_dihydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/c24_dihydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O4', 'delta_ref': 393.2999,
    },
    'c24_trihydroxy_combined': {
        'sirius': f'{BASE}/SIRIUS_outputs/c24_trihydroxy_combined.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/c24_trihydroxy_combined_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_trihydroxy_combined_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O5', 'delta_ref': 409.2949,
    },
    'c24_tetrahydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_tetrahydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/c24_tetrahydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O6', 'delta_ref': 425.2898,
    },
    'c24_pentahydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_pentahydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/c24_pentahydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_pentahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O7', 'delta_ref': 441.2847,
    },

    'C23_monohydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C23_monohydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C23_monohydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C23_monohydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C23H38O3', 'delta_ref': 363.2894,
    },
    'C23_dihydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C23_dihydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C23_dihydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C23_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C23H38O4', 'delta_ref': 379.2843,
    },
    'C23_trihydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C23_trihydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C23_trihydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C23_trihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C23H38O5', 'delta_ref': 395.2792,
    },
    'C23_tetrahydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C23_tetrahydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C23_tetrahydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C23_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C23H38O6', 'delta_ref': 411.2741,
    },
    'C23_pentahydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C23_pentahydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C23_pentahydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C23_pentahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C23H38O7', 'delta_ref': 427.269,
    },

    'C27_monohydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C27_monohydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C27_monohydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C27_monohydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C27H46O3', 'delta_ref': 419.352,
    },
    'C27_dihydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C27_dihydroxy_output_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C27_dihydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C27_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C27H46O4', 'delta_ref': 435.3469,
    },
    'C27_trihydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C27_trihydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C27_trihydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C27_trihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C27H46O5', 'delta_ref': 451.3418,
    },
    'C27_tetrahydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C27_tetrahydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C27_tetrahydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C27_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C27H46O6', 'delta_ref': 467.3367,
    },
    'C27_pentahydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C27_pentahydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C27_pentahydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C27_pentahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C27H46O7', 'delta_ref': 483.3316,
    },

    'Aceto_dihydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C26_aceto_dihydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/Aceto_dihydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/Aceto_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C26H42O5', 'delta_ref': 435.3105,
    },

    'C24_alkylamine_monohydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_alkamine_monohydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C24_alkylamine_monohydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_monohydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O3', 'delta_ref': 377.305,
    },
    'C24_alkylamine_dihydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_alkamine_dihydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C24_alkylamine_dihydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O4', 'delta_ref': 393.2999,
    },
    'C24_alkylamine_trihydroxy_combined': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_alkamine_trihydroxy_combined.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C24_alkylamine_trihydroxy_combined_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_trihydroxy_combined_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O5', 'delta_ref': 409.2949,
    },
    'C24_alkylamine_tetrahydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_alkamine_tetrahydroxy_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C24_alkylamine_tetrahydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O6', 'delta_ref': 425.2898,
    },
    'C24_alkylamine_pentahydroxy': {
        'sirius': f'{BASE}/SIRIUS_outputs/C24_alkylamine_pentahydroxy_output_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C24_alkylamine_pentahydroxy_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_pentahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': 'C24H40O7', 'delta_ref': 441.2847,
    },

    # ---- C28 alcohols: no core formula to subtract, no delta column ----
    'C28_alcohol_1OH': {
        'sirius': f'{BASE}/SIRIUS_outputs/C28_alcohol_1OH_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C28_alcohol_1OH_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C28_alcohol_1OH_falcon_eps01_minsamples2_scan_annotation_table.tsv',
        'ba_formula': None, 'delta_ref': None,
    },
    'C28_alcohol_2OH': {
        'sirius': f'{BASE}/SIRIUS_outputs/C28_alcohol_2OH_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C28_alcohol_2OH_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C28_alcohol_2OH_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': None, 'delta_ref': None,
    },
    'C28_alcohol_3OH': {
        'sirius': f'{BASE}/SIRIUS_outputs/C28_alcohol_3OH_summaries/formula_identifications.tsv',
        'buddy': f'{BASE}/BUDDY_outputs/C28_alcohol_3OH_msbuddy_result_summary.tsv',
        'annotation': f'{BASE}/final_mgfs_library_generation/individual_files_mgf/C28_alcohol_3OH_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
        'ba_formula': None, 'delta_ref': None,
    },
}

In [ ]:
def format_molecular_formula(formula):
    # Use regex to extract the element symbols and counts
    matches = re.findall(r'([A-Z][a-z]*)(-?\d*)', formula)

    # Create a dictionary to store the counts for each element
    counts = {}
    for match in matches:
        element, count = match
        if element == '':
            continue
        count = int(count) if count else 1
        counts[element] = counts.get(element, 0) + count

    # Format the counts as a comma-separated string
    formatted_counts = ', '.join([f'{count}{element}' for element, count in counts.items()])

    return formatted_counts

In [ ]:
import pandas as pd
import chemparse
import re

for core, cfg in CONFIG.items():

    delta_ref = cfg['delta_ref']
    ba_formula = cfg['ba_formula']

    #First, we import all the predicted data from BUDDY and SIRIUS
    df_buddy = pd.read_csv(cfg['buddy'], sep='\t')

    df_sirius = pd.read_csv(cfg['sirius'], sep='\t')
    df_sirius['identifier'] = df_sirius['mappingFeatureId'].str.split('UNKNOWN_').str[-1]

    if delta_ref is not None:
        #calculating values for a new column to be populated by delta masses.
        df_buddy['delta_mass'] = (df_buddy['mz'] - delta_ref)
        df_sirius['delta_mass'] = (df_sirius['ionMass'] - delta_ref)

        #round to 4 decimals
        df_buddy['delta_mass'] = df_buddy['delta_mass'].round(decimals=4)
        df_sirius['delta_mass'] = df_sirius['delta_mass'].round(decimals=4)

        #add the bile acid formula of the core relative to the imported files
        df_buddy['Formula_BA_buddy'] = ba_formula
        df_sirius['Formula_BA_sirius'] = ba_formula

        #Now we subtract the prediction of BUDDY (neutral) with the respective neutral BA formula
        formula_buddy = []

        for i in range(len(df_buddy)):
            # Convert values to strings and check for 'None' or empty strings
            formula_rank_1 = str(df_buddy['formula_rank_1'][i])
            formula_BA_buddy = str(df_buddy['Formula_BA_buddy'][i])

            if formula_rank_1 in ['nan', ''] or formula_BA_buddy in ['nan', '']:
                formula_buddy.append("N/A")
                continue

            formula_final = ''
            dict_formula = chemparse.parse_formula(formula_rank_1)
            dict_formula_BA = chemparse.parse_formula(formula_BA_buddy)

            for element in dict_formula.keys():
                try:
                    if dict_formula[element] - dict_formula_BA[element] == 0:
                        continue
                    else:
                        formula_final = formula_final + element + str(dict_formula[element] - dict_formula_BA[element]).split('.')[0]
                except KeyError:
                    formula_final = formula_final + element + str(dict_formula[element]).split('.')[0]

            formula_buddy.append(formula_final)

        df_buddy['atomic_difference_buddy'] = formula_buddy
        df_buddy['atomic_difference_buddy'].loc[(df_buddy['formula_rank_1'] == 'N/A')] = 'N/A'
        df_buddy['atomic_difference_buddy'].loc[(df_buddy['formula_rank_1'] == df_buddy['Formula_BA_buddy'])] = '0'

        #Now we subtract the prediction of SIRIUS (neutral) with the respective neutral BA formula
        formula_sirius = []

        for i in range(len(df_sirius)):
            # Convert values to strings and check for 'None' or empty strings
            formula_rank_1 = str(df_sirius['molecularFormula'][i])
            formula_BA_sirius = str(df_sirius['Formula_BA_sirius'][i])

            if formula_rank_1 in ['nan', ''] or formula_BA_sirius in ['nan', '']:
                formula_sirius.append("N/A")
                continue

            formula_final = ''
            dict_formula = chemparse.parse_formula(formula_rank_1)
            dict_formula_BA = chemparse.parse_formula(formula_BA_sirius)

            for element in dict_formula.keys():
                try:
                    if dict_formula[element] - dict_formula_BA[element] == 0:
                        continue
                    else:
                        formula_final = formula_final + element + str(dict_formula[element] - dict_formula_BA[element]).split('.')[0]
                except KeyError:
                    formula_final = formula_final + element + str(dict_formula[element]).split('.')[0]

            formula_sirius.append(formula_final)

        df_sirius['atomic_difference_sirius'] = formula_sirius
        df_sirius['atomic_difference_sirius'].loc[(df_sirius['molecularFormula'] == 'N/A')] = 'N/A'
        df_sirius['atomic_difference_sirius'].loc[(df_sirius['molecularFormula'] == df_sirius['Formula_BA_sirius'])] = '0'

    ## FILTERS BUDDY AND SIRIUS

    #convert float to numeric and filter to only results that have fdr in buddy < 0.05
    df_buddy['estimated_fdr'] = pd.to_numeric(df_buddy['estimated_fdr'], errors='coerce')
    df_buddy = df_buddy[df_buddy['estimated_fdr'] < 0.05]

    df_sirius['SiriusScoreNormalized'] = pd.to_numeric(df_sirius['SiriusScoreNormalized'], errors='coerce')
    df_sirius = df_sirius[(df_sirius['SiriusScoreNormalized'] > 0.9) & (df_sirius['numExplainedPeaks'].abs() > 4)]

    #now we can merge these two tables
    df_buddy_sirius_merged = pd.merge(df_buddy, df_sirius, on='identifier', how='outer')
    df_buddy_sirius_merged.fillna('N/A', inplace=True)
    df_buddy_sirius_merged = df_buddy_sirius_merged.rename(columns={'adduct_x':'adduct_buddy',
                                                                    'adduct_y': 'adduct_sirius',
                                                                    'delta_mass_x': 'delta_mass_buddy',
                                                                    'delta_mass_y': 'delta_mass_sirius',
                                                                    'formula_rank_1':'Formula_buddy',
                                                                    'molecularFormula': 'Formula_sirius'})

    merged_path = f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/{core}_buddy_sirius_merged.tsv'
    df_buddy_sirius_merged.to_csv(merged_path, sep='\t', index=False)

    #import tables
    annotation_table = pd.read_csv(cfg['annotation'], sep='\t')
    annotation_table.rename(columns={'title': 'identifier'}, inplace=True) #just to be sure that the column is named the same in both tables for the merge

    sirius_buddy = pd.read_csv(merged_path, sep='\t')

    #merge and keep only columns of interest
    sirius_buddy = pd.merge(annotation_table, sirius_buddy, on='identifier', how='left')

    if delta_ref is not None:
        sirius_buddy = sirius_buddy[['identifier', 'pepmass', 'Formula_buddy', 'atomic_difference_buddy',
                                           'Formula_sirius', 'atomic_difference_sirius']]

        #add information of the unconjugated BA and calculate the deltas:
        sirius_buddy['delta_mass'] = sirius_buddy['pepmass'] - delta_ref
        sirius_buddy['delta_mass'] = sirius_buddy['delta_mass'].round(decimals=4)
        sirius_buddy.fillna('N/A', inplace=True)

        #let's apply to all the atoms differences columns
        formatted_formulas_buddy = [format_molecular_formula(f) for f in sirius_buddy['atomic_difference_buddy']]
        sirius_buddy['atomic_difference_buddy_ok'] = formatted_formulas_buddy

        formatted_formulas_sirius = [format_molecular_formula(f) for f in sirius_buddy['atomic_difference_sirius']]
        sirius_buddy['atomic_difference_sirius_ok'] = formatted_formulas_sirius

        # and we can create a new column with what we want to put in the 'name', in which the core comes from the dictionary key
        sirius_buddy['annotation'] = sirius_buddy.apply(lambda x: f"Candidate bile acid {core} (delta mass {x['delta_mass']}; sirius atomic difference: {x['atomic_difference_sirius_ok']}; buddy atomic difference: {x['atomic_difference_buddy_ok']})", axis=1)

        #making small replaces to not account for the N/As in the annotation column or no modification
        sirius_buddy['annotation'] = sirius_buddy['annotation'].replace({'; buddy atomic difference: 1N, 1A': ''}, regex=True)
        sirius_buddy['annotation'] = sirius_buddy['annotation'].replace({'; sirius atomic difference: 1N, 1A': ''}, regex=True)
        sirius_buddy['annotation'] = sirius_buddy['annotation'].replace({'; sirius atomic difference: ; buddy atomic difference: ': ''}, regex=True)

        #remove any potential duplicated rows
        sirius_buddy = sirius_buddy.drop_duplicates()

    else:
        # C28 alcohols: no delta, keep only the SIRIUS and BUDDY predicted formulas
        sirius_buddy = sirius_buddy[['identifier', 'pepmass', 'Formula_buddy', 'Formula_sirius']]
        sirius_buddy.fillna('N/A', inplace=True)

        sirius_buddy['annotation'] = sirius_buddy.apply(lambda x: f"Candidate bile acid {core} (sirius formula: {x['Formula_sirius']}; buddy formula: {x['Formula_buddy']})", axis=1)

        sirius_buddy = sirius_buddy.drop_duplicates()

    sirius_buddy.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/{core}_buddy_sirius_merged_final_annotation.tsv', sep='\t', index=False)

    print(f'done: {core}')